# Checkpoint de entrega — treino final (política P1)

Roda **um** ajuste com TODAS as pessoas do MINDS, política `--politica-final ultima` (sem avaliação sobreposta ao treino — só a última época, nunca a melhor numa validação que também treinou), semente e backbone fixados conforme [`docs/politica-modelo-final-2026-09-14.md`](../../docs/politica-modelo-final-2026-09-14.md) — não escolhidos por desempenho.

**Não é** LOSO, não gera relatório de acurácia, não é avaliação independente. O checkpoint resultante (`modelo_final.pt`) ainda precisa de exportação e paridade próprias — não herda a aprovação do piloto M01.


## 1. Ambiente e código


In [ ]:
import os, pathlib, shutil, subprocess, sys, tarfile

EM_KAGGLE = pathlib.Path("/kaggle/working").is_dir() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
BASE = pathlib.Path("/kaggle/working" if EM_KAGGLE else ".").resolve()
URL = "https://github.com/Heitorvazeg/libras-livre-ai-glasses-brasil.git"
BRANCH = "feat/treino-final-politica"

def git(*args, repo=None):
    cmd = ["git"] + (["-C", str(repo)] if repo else []) + list(args)
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except subprocess.CalledProcessError as erro:
        raise RuntimeError(f"Git falhou: {erro.stderr.strip()}. No Kaggle, habilite Internet; "
                           "não prosseguir com código desatualizado.") from erro

if EM_KAGGLE:
    REPO = BASE / "libras-livre-ai-glasses-brasil"
    if REPO.exists():
        if git("branch", "--show-current", repo=REPO) != BRANCH:
            raise RuntimeError("Clone em outra branch; use outro runtime. Nada será apagado.")
        if git("status", "--porcelain", "--untracked-files=no", repo=REPO):
            raise RuntimeError("Clone tem alterações locais; nada será sobrescrito.")
        git("fetch", "origin", BRANCH, repo=REPO)
        git("merge", "--ff-only", "FETCH_HEAD", repo=REPO)
    else:
        git("clone", "--single-branch", "--branch", BRANCH, URL, str(REPO))
else:
    REPO = next((p for p in (BASE, *BASE.parents)
                 if (p / "computer-vision-model" / "treino").is_dir()), None)
    if REPO is None:
        raise RuntimeError("Execute localmente dentro do repositório.")

TREINO = (REPO / "computer-vision-model" / "treino").resolve()
if not (TREINO / "evidencias_loso.py").is_file():
    raise RuntimeError("Este clone não tem a política final (evidencias_loso.py ausente). "
                       "Publique/atualize a branch feat/treino-final-politica.")
COMMIT_CODIGO = git("rev-parse", "HEAD", repo=REPO)
print("ambiente:", "Kaggle" if EM_KAGGLE else "local")
print("código:", COMMIT_CODIGO, "|", TREINO)


In [ ]:
import importlib.util
faltantes = [pacote for modulo, pacote in (("yaml", "pyyaml"), ("scipy", "scipy"))
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes], check=True)
import torch
if EM_KAGGLE and not torch.cuda.is_available():
    raise RuntimeError("Accelerator = GPU não está ligado nesta sessão.")
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())


## 2. Insumos: MINDS (para treinar) e o backbone já verificado (para inicializar)

Anexe dois datasets privados:
- `landmarks-minds.tar.gz` → `landmarks/` — o mesmo pacote das outras runs.
- `backbone-final.tar.gz` (ou pasta equivalente) contendo `backbone_gcn.pt` — o backbone do piloto M01, **hash conferido abaixo antes de qualquer uso**. Não é o backbone da PoC de negativos extras nem de WLASL.


In [ ]:
sys.path.insert(0, str(TREINO))
import entrada_pretreino as entrada

HASH_BACKBONE_APROVADO = "7a6e997c5830139162b32bc9b37a48e6eb5b8d6222836da87cb95e94ecf6baf5"

RAIZ_INPUT = pathlib.Path("/kaggle/input") if EM_KAGGLE else BASE
DESTINO = BASE / "dados-treino-final"
DESTINO.mkdir(parents=True, exist_ok=True)

# MINDS: reaproveita a busca já usada no notebook de pré-treino (tar.gz OU pasta
# já extraída, em qualquer slug/profundidade).
origem_minds = entrada.localizar(RAIZ_INPUT, "minds", None)
if origem_minds.is_file():
    with tarfile.open(origem_minds, "r:gz") as tar:
        tar.extractall(DESTINO, filter="data")
    MINDS = DESTINO / "landmarks"
else:
    MINDS = origem_minds
if not any(MINDS.glob("pessoaM*.npy")):
    raise RuntimeError(f"Nenhum landmark MINDS em {MINDS}.")
print("MINDS:", MINDS, "|", len(list(MINDS.glob('pessoaM*.npy'))), "clipes")

# Backbone: procura um backbone_gcn.pt em qualquer lugar do Input (tar.gz OU
# arquivo já extraído), sem adivinhar entre candidatos genuinamente diferentes.
candidatos = sorted(set(RAIZ_INPUT.rglob("backbone_gcn.pt")))
if len(candidatos) == 1:
    BACKBONE = candidatos[0]
else:
    tars = sorted(set(RAIZ_INPUT.rglob("*.tar.gz")) & set(RAIZ_INPUT.rglob("*backbone*")))
    if len(tars) != 1:
        raise RuntimeError(f"Esperado 1 backbone_gcn.pt (achado {len(candidatos)}) ou 1 "
                           f"tar.gz com 'backbone' no nome (achado {len(tars)}). "
                           "Anexe o dataset do backbone explicitamente.")
    with tarfile.open(tars[0], "r:gz") as tar:
        tar.extractall(DESTINO / "backbone", filter="data")
    achados = list((DESTINO / "backbone").rglob("backbone_gcn.pt"))
    if len(achados) != 1:
        raise RuntimeError(f"tar.gz do backbone não contém exatamente 1 backbone_gcn.pt "
                           f"(achado {len(achados)}).")
    BACKBONE = achados[0]

hash_real = entrada.pv.hash_arquivo(BACKBONE)
if hash_real != HASH_BACKBONE_APROVADO:
    raise RuntimeError(f"Backbone com hash {hash_real}, esperado {HASH_BACKBONE_APROVADO}. "
                       "Não é o backbone aprovado pela política P1 — não usar.")
print("Backbone verificado:", BACKBONE, "|", hash_real)


## 3. Selftest — obrigatório antes do treino longo


In [ ]:
def executar(script, argumentos, log):
    comando = [sys.executable, "-u", script, *argumentos]
    print("Executando:", " ".join(comando), flush=True)
    with (EXP / log).open("a", encoding="utf-8") as arquivo_log:
        with subprocess.Popen(comando, cwd=TREINO, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as processo:
            for linha in processo.stdout:
                print(linha, end="", flush=True)
                arquivo_log.write(linha)
                arquivo_log.flush()
            if processo.wait():
                raise subprocess.CalledProcessError(processo.returncode, comando)

NOME_EXPERIMENTO = "final-s20260917-v1"
EXP = BASE / "experimentos-privados" / NOME_EXPERIMENTO
EXP.mkdir(parents=True, exist_ok=True)

executar("selftest.py", [], "selftest.log")
print("Selftest passou. Saídas privadas:", EXP)


## 4. O treino final

`--politica-final ultima` exige `--semente` explícita e recusa saída não-vazia — não sobrescreve uma tentativa anterior por engano. Sem avaliação durante o treino: perda/acurácia no log são diagnóstico, não seleção.


In [ ]:
SAIDA_FINAL = EXP / "modelo_final"
FINAL_ARGS = [
    "--arquitetura", "gcn", "--ossos", "--com-z", "--z-recentrado",
    "--fontes", "minds", "--landmarks", str(MINDS),
    "--inicializar", str(BACKBONE),
    "--epocas", "120", "--lr", "1e-3", "--wd", "1e-4", "--batch", "64",
    "--agendador", "cosseno",
    "--final", "--politica-final", "ultima", "--semente", "20260917",
    "--dispositivo", "cuda", "--threads", "4", "--saida", str(SAIDA_FINAL),
]
executar("treinar.py", FINAL_ARGS, "treino-final.log")

destino = SAIDA_FINAL / "modelo_final.pt"
if not destino.is_file():
    raise RuntimeError("treinar.py terminou sem gerar modelo_final.pt.")
print("Checkpoint final:", destino, "|", entrada.pv.hash_arquivo(destino))


## 5. Backup privado


In [ ]:
def empacotar():
    arquivo = EXP.parent / f"{EXP.name}.tar.gz"
    parcial = arquivo.with_suffix(".parcial")
    with tarfile.open(parcial, "w:gz") as tar:
        tar.add(EXP, arcname=EXP.name)
    parcial.replace(arquivo)
    return arquivo

arquivo = empacotar()
print("Arquivo privado:", arquivo, "|", arquivo.stat().st_size, "bytes")
print("Kaggle: Save Version e mantenha notebook, datasets e outputs PRIVADOS.")
